# Location Selection with E-NAUTILUS: Part 2
_Running of E-NAUTILUS for decision making, and presentation of results_


In [1]:
import numpy as np
import pandas as pd
import polars as pl
import pickle
import folium
# These are to just suppress warnings in the outputs of the example
import warnings

warnings.filterwarnings("ignore")

## Load results from previous session

In [2]:
file_name = "data/pf_16.pkl"

output = open(file_name, 'rb')
prev_session = pickle.load(output)

raw_ref_pf = prev_session["pf"]
prob = prev_session["prob"]
events = prev_session["events"]
cities = prev_session["cities"]
event2city = prev_session["event2city"]


## Load reference front and problem 

In [3]:
output_flat = np.array(raw_ref_pf).flatten()

def process_lists(dict2conv):
    return {key: np.array(dict2conv[key]).flatten().tolist() for key in dict2conv.keys()}

# TODO include constraints in here too
output_dict = [
    output.optimal_objectives | 
    process_lists(output.optimal_variables) 
    for output in output_flat]

nd_df = pl.DataFrame(output_dict)

nd_df = nd_df.unique(subset=("f_1", "f_2", "f_3", "f_4"))

nd_df = nd_df.with_columns([
    (-pl.col("f_1")).alias("f_1_min"),
    (pl.col("f_2")).alias("f_2_min"),
    (pl.col("f_3")).alias("f_3_min"),
    (-pl.col("f_4")).alias("f_4_min")
])

nadir_point = {
  "f_1": float(nd_df["f_1"].min()),
  "f_2": float(nd_df["f_2"].max()),
  "f_3": float(nd_df["f_3"].max()),
  "f_4": float(nd_df["f_4"].min())
}

display(nd_df)

print(f"Nadir point: {nadir_point}")
print(f"Nadir point (problem): {prob.get_nadir_point()}")
print(f"Idedal point (problem): {prob.get_ideal_point()}")


reachable_indices = list(range(len(nd_df)))  # everything reachable from nadir


f_1,f_2,f_3,f_4,ev,cover,_alpha,f_1_min,f_2_min,f_3_min,f_4_min
f64,f64,f64,f64,list[f64],list[f64],list[f64],f64,f64,f64,f64
0.0,0.0,0.0,0.0,"[0.0, 0.0, … 0.0]","[0.0, 0.0, … 0.0]",[-9.9600e-9],-0.0,0.0,0.0,-0.0
49692.0,0.0,492.02,0.516395,"[0.0, 0.0, … 1.0]","[0.0, 0.0, … 0.0]",[0.004567],-49692.0,0.0,492.02,-0.516395
49596.0,3.0,665.5,0.682044,"[1.0, 0.0, … 1.0]","[1.0, 1.0, … 0.0]",[0.317955],-49596.0,3.0,665.5,-0.682044
49479.0,0.0,224.6,0.269346,"[0.0, 0.0, … 1.0]","[0.0, 0.0, … 0.0]",[0.123091],-49479.0,0.0,224.6,-0.269346
49919.0,9.0,1744.26,0.712359,"[1.0, 1.0, … 1.0]","[1.0, 1.0, … 0.0]",[-0.007701],-49919.0,9.0,1744.26,-0.712359
…,…,…,…,…,…,…,…,…,…,…
49599.0,6.0,937.925,0.712359,"[1.0, 0.0, … 1.0]","[1.0, 1.0, … 0.0]",[707.261843],-49599.0,6.0,937.925,-0.712359
49879.0,0.0,935.785,0.565712,"[0.0, 0.0, … 1.0]","[0.0, 0.0, … 0.0]",[0.018646],-49879.0,0.0,935.785,-0.565712
49565.0,0.0,402.235,0.565712,"[0.0, 0.0, … 1.0]","[0.0, 0.0, … 0.0]",[-9.9900e-9],-49565.0,0.0,402.235,-0.565712


Nadir point: {'f_1': 0.0, 'f_2': 10.0, 'f_3': 1824.665, 'f_4': 0.0}
Nadir point (problem): {'f_1': 0, 'f_2': 10, 'f_3': 1824.665, 'f_4': 0}
Idedal point (problem): {'f_1': 49920, 'f_2': 0, 'f_3': 0, 'f_4': 1.0}


## Helper functions

In [4]:
# Function to determine marker size based on population
def get_marker_size(population):
    return max(5, population / 1000)  # Adjust the divisor to scale marker size

def create_color_dict(cities, ev_cities, cc): 
    marker_color = {}
    for city in cities.loc[:,"city"]: 
        if city in ev_cities: 
            marker_color[city] = "orange"
        elif city in cc: 
            marker_color[city] = "yellow"
        else: 
            marker_color[city] = "grey"

    return marker_color

def select_point(results, sol_id): 

    return {
        "f_1": int(results.loc[sol_id, "Total patients served"]),
        "f_2": int(results.loc[sol_id, "Number of overstaffed events"]),
        "f_3": float(results.loc[sol_id, "Total costs ($)"]),
        "f_4": float(results.loc[sol_id, "Population with access (%)"]/100.0)
        }


def clean_results(raw_results, intermediate_point=True): 
    # Transform objectives
    if intermediate_point: 
        results = pd.DataFrame(raw_results.intermediate_points)
    else:
        results = pd.DataFrame(raw_results.optimal_objectives)

    results = results.rename(columns={
                    "f_1": "Total patients served", 
                    "f_2": "Number of overstaffed events", 
                    "f_3": "Total costs ($)", 
                    "f_4": "Population with access (%)"})
    results[["Total patients served", "Number of overstaffed events"]] =  results[["Total patients served", "Number of overstaffed events"]].astype(int)
    results[["Population with access (%)"]] = (results[["Population with access (%)"]]*100.0).round(2)
    results[["Total costs ($)"]] = (results[["Total costs ($)"]]).round(2)

    results.index.name = "Solution ID"

    return results

## Run eNAUTILUS 
### Round 1
We're going to generate some solutions. They will be poor at first, but you and the computer will slowly find the best solution that fulfills your goals and preferences. 


In [9]:
from desdeo.mcdm.enautilus import enautilus_step
from desdeo.mcdm.enautilus import enautilus_get_representative_solutions

current_iter = 0
selected_point = nadir_point
total_iterations = 3
display(f"Starting with point {selected_point}")

prob.get_ideal_point()


raw_results = enautilus_step(
    problem=prob,
    non_dominated_points=nd_df,
    current_iteration=current_iter,
    iterations_left=total_iterations - current_iter,
    selected_point=selected_point,
    reachable_point_indices=reachable_indices,
    total_number_of_iterations=total_iterations,
    number_of_intermediate_points=3,
)

print(f"number of iterations left: {total_iterations - current_iter}")

results = clean_results(raw_results)
display("Which solution to do you prefer?")
display(results)

"Starting with point {'f_1': 0.0, 'f_2': 10.0, 'f_3': 1824.665, 'f_4': 0.0}"

number of iterations left: 3


'Which solution to do you prefer?'

,Total patients served,Number of overstaffed events,Total costs ($),Population with access (%)
Solution ID,,,,
0,16640,10,1824.66,23.60
1,16532,7,1438.28,22.73
2,0,6,1216.44,0.00


### Round 2 

In [10]:
chosen_solution = 1

current_iter += 1
selected_point = select_point(results, chosen_solution)

print(selected_point)

raw_results = enautilus_step(
    problem=prob,
    non_dominated_points=nd_df,
    current_iteration=current_iter,
    iterations_left=total_iterations - current_iter,
    selected_point=selected_point,
    reachable_point_indices=reachable_indices,
    total_number_of_iterations=total_iterations,
    number_of_intermediate_points=3,
)

print(f"number of iterations left: {total_iterations - current_iter}")
display(raw_results)
results = clean_results(raw_results)
display("Which solution to do you find most preferable?")
display(results)
display("Results:")



{'f_1': 16532, 'f_2': 7, 'f_3': 1438.28, 'f_4': 0.2273}
number of iterations left: 2


ENautilusResult(current_iteration=2, iterations_left=1, intermediate_points=[{'f_1': 33226.0, 'f_2': 8.5, 'f_3': 1631.4724999999999, 'f_4': 0.46763902831043425}, {'f_1': 33064.0, 'f_2': 5.0, 'f_3': 1051.8899999999999, 'f_4': 0.4546722040188157}, {'f_1': 8266.0, 'f_2': 3.5, 'f_3': 719.14, 'f_4': 0.11365}], reachable_best_bounds=[{'f_1': 49879.0, 'f_2': 0.0, 'f_3': 402.235, 'f_4': 0.7123592855990368}, {'f_1': 49879.0, 'f_2': 0.0, 'f_3': 402.235, 'f_4': 0.6820444080376314}, {'f_1': 49692.0, 'f_2': 0.0, 'f_3': 224.6, 'f_4': 0.6820444080376314}], reachable_worst_bounds=[{'f_1': 33226.0, 'f_2': 8.5, 'f_3': 1631.4724999999999, 'f_4': 0.46763902831043425}, {'f_1': 33064.0, 'f_2': 5.0, 'f_3': 1051.8899999999999, 'f_4': 0.4546722040188157}, {'f_1': 8266.0, 'f_2': 3.5, 'f_3': 719.14, 'f_4': 0.11365}], closeness_measures=[66.55961876548959, 66.66666668636148, 457.04159921291864], reachable_point_indices=[[1, 2, 5, 6, 7, 8, 10], [1, 2, 5, 7, 8, 10], [1, 2, 3, 8, 10]])

'Which solution to do you find most preferable?'

,Total patients served,Number of overstaffed events,Total costs ($),Population with access (%)
Solution ID,,,,
0,33226,8,1631.47,46.76
1,33064,5,1051.89,45.47
2,8266,3,719.14,11.36


'Results:'

### Round 3

In [11]:

chosen_solution = 1

current_iter += 1
selected_point = select_point(results, chosen_solution)

print(selected_point)

raw_results = enautilus_step(
    problem=prob,
    non_dominated_points=nd_df,
    current_iteration=current_iter,
    iterations_left=total_iterations - current_iter,
    selected_point=selected_point,
    reachable_point_indices=reachable_indices,
    total_number_of_iterations=total_iterations,
    number_of_intermediate_points=3,
)

print(f"number of iterations left: {total_iterations - current_iter}")

results = clean_results(raw_results)
display("Which solution to do you find most preferable?")
display(results)




{'f_1': 33064, 'f_2': 5, 'f_3': 1051.89, 'f_4': 0.4547}
number of iterations left: 1


'Which solution to do you find most preferable?'

,Total patients served,Number of overstaffed events,Total costs ($),Population with access (%)
Solution ID,,,,
0,49920,10,1824.66,70.8
1,49596,3,665.50,68.2
2,0,0,0.00,0.0


## Display final result

In [13]:
final_chosen_solution = 1

solutions = enautilus_get_representative_solutions(prob, raw_results, nd_df) 
solution = solutions[final_chosen_solution]
results = clean_results(solution, intermediate_point=False)

display(results)
 


,Total patients served,Number of overstaffed events,Total costs ($),Population with access (%)
Solution ID,,,,
0,49596,3,665.5,68.2


### Result postprocessing

In [14]:

raw_events = solution.optimal_variables['ev'][0].to_list()
raw_events = [[bool(e) for e in raw_events]]

raw_coverage = solution.optimal_variables['cover'][0].to_list()
raw_coverage = [[bool(c) for c in raw_coverage]]

events_visited = []
for evb in raw_events: 
    events_visited.append("\n".join(events.loc[evb, "event_id"].values))

cities_covered = [] 
for cc in raw_coverage: 
    cities_covered.append("\n".join(cities.loc[cc,"city"].values))

results["Events Visited"] = events_visited
results["Cities covered"] = cities_covered

results

,Total patients served,Number of overstaffed events,Total costs ($),Population with access (%),Events Visited,Cities covered
Solution ID,,,,,,
0,49596,3,665.5,68.2,ada-public-library\nbluffton-bluffton-public-l...,Ada\nAlger\nBluffton\nCairo\nCridersville\nDel...


### Map preprocessing

In [15]:
events_in_cities = events.loc[raw_events[0],:].groupby("city").agg({"event_pretty": lambda e : '<br>'.join(e)})
events_in_cities = events_in_cities.to_dict()['event_pretty']
events_in_cities

# cc cities covered
cc = set(cities_covered[0].split('\n'))

# event_cities 
ev_cities = list(events.loc[raw_events[0], "city"])
marker_colors = create_color_dict(cities, ev_cities, cc)


## Event coverage dictionary 
event2city_mat = event2city[raw_events[0]].astype(bool)

event2city_dict = {}
for (c,city) in enumerate(ev_cities): 
    event2city_dict[city] = set(cities.loc[event2city_mat[c,],"city"]) - {city}

adjacent_events = {}
# Record what events are near cities
for event_city in event2city_dict.keys(): 
    adj_cities = event2city_dict[event_city]
    from_name = cities.loc[cities.loc[:,"city"] == event_city,["city"]].values.tolist()[0][0]

    for adj_city in adj_cities: 
        to_name = cities.loc[cities.loc[:,"city"] == adj_city ,["city"]].values.tolist()[0][0]

        if to_name not in adjacent_events.keys(): 
            adjacent_events[to_name] = {from_name}
        else: 
            adjacent_events[to_name] = adjacent_events[to_name].union({from_name})



## Render map

In [16]:
# Create a base map
m = folium.Map(location=[cities['lat'].mean(), 
                         cities['long'].mean()], 
                         zoom_start=7) 

# Draw lines between 
for event_city in event2city_dict.keys(): 
    adj_cities = event2city_dict[event_city]
    from_loc = cities.loc[cities.loc[:,"city"] == event_city,["lat", "long"]].values.tolist()

    for adj_city in adj_cities: 
        to_loc = cities.loc[cities.loc[:,"city"] == adj_city ,["lat", "long"]].values.tolist()
        folium.PolyLine(
            locations=[to_loc[0], from_loc[0]],
            color="black"
        ).add_to(m)

# Set bounds
sw = cities.loc[:,['lat', 'long']].min().values.tolist()
ne = cities.loc[:,['lat', 'long']].max().values.tolist()
m.fit_bounds([sw,ne])

# Create tool tips 
tooltips = {}
for _, row in cities.iterrows():
    city = row['city']
    tooltips[city]=f"<b>{city}</b><br><b>Population:</b> {row['pop']}"

    if city in events_in_cities.keys():
        tooltips[city]+= "<br><b>Events:</b><br>"
        tooltips[city]+= events_in_cities[city]
    else:
        tooltips[city]+= "<br><b>No Healthwise Clinics</b>"

    if city in adjacent_events.keys(): 
        tooltips[city]+= "<br><b>Covered by events in: </b>"
        tooltips[city]+= "<br>".join(adjacent_events[city])


# Add cities to the map
for _, row in cities.iterrows():
    city = row['city']
    folium.CircleMarker(
        location=(row['lat'], row['long']),
        radius=get_marker_size(row['pop']),
        color="black",
        fill=True,
        fill_color=marker_colors[city],
        fill_opacity=0.6,
        tooltip=tooltips[city]
    ).add_to(m)

display(results)
display(m)

,Total patients served,Number of overstaffed events,Total costs ($),Population with access (%),Events Visited,Cities covered
Solution ID,,,,,,
0,49596,3,665.5,68.2,ada-public-library\nbluffton-bluffton-public-l...,Ada\nAlger\nBluffton\nCairo\nCridersville\nDel...
